In [ ]:
# ==============================================================================
# Block 1: Model Architecture + Model Loading
# ==============================================================================

import torch
import torch.nn as nn
import numpy as np


if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

WL_FULL = np.arange(200, 801, 1)


class AutoregressiveTokenDecoder(nn.Module):
    def __init__(
        self,
        input_dim=601,
        num_layers=10,
        vocab_size=292,
        d_model=256,
        nhead=8,
        num_layers_dec=6,
        dim_feedforward=1024,
        dropout=0.05,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.num_layers = num_layers
        self.d_model = d_model

        self.spec_conv = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Conv1d(128, d_model, kernel_size=3, padding=1),
        )

        self.spec_pos_emb = nn.Parameter(torch.randn(1, input_dim, d_model) * 0.02)

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Parameter(torch.randn(1, num_layers + 1, d_model) * 0.02)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
        )

        self.transformer_decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=num_layers_dec,
        )

        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, spectrum, tgt_tokens):
        _, seq_len = tgt_tokens.size()

        spec_feats = self.spec_conv(spectrum.unsqueeze(1)).transpose(1, 2)
        spec_feats = spec_feats + self.spec_pos_emb

        tgt_emb = self.token_emb(tgt_tokens) + self.pos_emb[:, :seq_len, :]

        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(spectrum.device)

        out = self.transformer_decoder(
            tgt=tgt_emb,
            memory=spec_feats,
            tgt_mask=causal_mask,
        )

        return self.head(out)


def load_trained_model(
    ckpt_path="thin_film_transformer_final.pth",
    device=DEVICE,
):
    checkpoint = torch.load(
        ckpt_path,
        map_location=device,
        weights_only=False,
    )

    model_config = checkpoint["model_config"]

    model = AutoregressiveTokenDecoder(**model_config).to(device)

    model.load_state_dict(checkpoint["state_dict"])
    model.eval()

    print(
        f"Model loaded | "
        f"Input Dim: {model_config['input_dim']} | "
        f"Device: {device}"
    )

    return model, model_config

In [ ]:
# ==============================================================================
# Block 2: TMM Physics Engine + Random Test Structure
# ==============================================================================

import numpy as np
from simulation import Certain_Wavelength


FIXED_EPS = np.array(
    [3.9, 11.7, 3.9, 11.7, 3.9, 11.7, 3.9, 11.7, 3.9, 11.7],
    dtype=np.float32,
)


def calculate_transmittance_spectrum(thickness_nm, wavelengths_nm=WL_FULL, eps_in=1.0, eps_out=1.0):
    """
    使用與 Dataset Generation 相同的 TMM 計算 Transmittance Spectrum。

    thickness_nm : shape = (10,)
    return       : shape = (601,)
    """

    params = np.empty(20, dtype=np.float32)
    params[0::2] = FIXED_EPS
    params[1::2] = thickness_nm * 1e-9

    component = Certain_Wavelength(eps_in, eps_out, params)

    transmittance = []

    for wavelength_nm in wavelengths_nm:
        _, T, _ = component.RTA(wavelength_nm * 1e-9, eps_in, eps_out)
        transmittance.append(T)

    return np.asarray(transmittance, dtype=np.float32)


def generate_random_test_sample():
    """
    隨機生成全新的 10 層連續厚度結構。

    Thickness:
        10.0 ~ 300.0 nm
        使用連續均勻分布，不限制為整數 nm。

    Output:
        target_thickness_nm : shape = (10,)
        target_spectrum     : shape = (601,)
    """

    target_thickness_nm = np.random.uniform(10.0, 300.0, size=10).astype(np.float32)
    target_spectrum = calculate_transmittance_spectrum(target_thickness_nm)

    return target_thickness_nm, target_spectrum

In [ ]:
# ==============================================================================
# Block 3: 10-Layer Beam Search Inference
# ==============================================================================

import torch
import numpy as np


def run_beam_search_10_layers(model, target_spectrum, beam_width=24, device=DEVICE):
    """
    使用 Beam Search 進行固定 10 層薄膜反向設計。

    Token Mapping:
        Token 0     = <BOS>
        Token 1~291 = 10~300 nm

    Output:
        best_thickness_nm : shape = (10,)
        best_spectrum     : shape = (601,)
        best_spectrum_mae : float
    """

    model.eval()

    spec_tensor = torch.tensor(
        target_spectrum,
        dtype=torch.float32,
        device=device,
    ).unsqueeze(0)

    with torch.no_grad():

        # (累積 Log Probability, Token Sequence)
        sequences = [(0.0, [0])]

        for step in range(10):

            all_candidates = []

            for score, sequence in sequences:

                input_tokens = torch.tensor(
                    [sequence],
                    dtype=torch.long,
                    device=device,
                )

                logits = model(spec_tensor, input_tokens)
                next_logits = logits[0, -1, :].clone()

                # BOS 只能出現在序列開頭
                next_logits[0] = -float("inf")

                log_probs = torch.log_softmax(next_logits, dim=-1).cpu().numpy()
                top_tokens = np.argsort(log_probs)[-beam_width:]

                for token in top_tokens:
                    new_score = score + log_probs[token]
                    new_sequence = sequence + [token]

                    all_candidates.append(
                        (new_score, new_sequence)
                    )

            all_candidates.sort(
                key=lambda x: x[0],
                reverse=True,
            )

            sequences = all_candidates[:beam_width]


    """
    使用 TMM 重新計算 Beam 候選的實際 Transmittance Spectrum，
    最後選擇 Spectrum MAE 最低的結構。
    """

    best_spectrum_mae = float("inf")
    best_thickness_nm = None
    best_spectrum = None

    for score, sequence in sequences:

        tokens = np.asarray(sequence[1:11], dtype=np.float32)

        # Token 1 -> 10 nm
        # Token 291 -> 300 nm
        thickness_nm = (tokens - 1.0) + 10.0

        simulated_spectrum = calculate_transmittance_spectrum(thickness_nm)

        spectrum_mae = np.mean(
            np.abs(target_spectrum - simulated_spectrum)
        )

        if spectrum_mae < best_spectrum_mae:
            best_spectrum_mae = spectrum_mae
            best_thickness_nm = thickness_nm
            best_spectrum = simulated_spectrum

    return best_thickness_nm, best_spectrum, float(best_spectrum_mae)

In [ ]:
# ==============================================================================
# Block 4: 10-Layer L-BFGS-B Continuous Refinement
# ==============================================================================

import numpy as np
from scipy.optimize import minimize


def run_lbfgsb_10_layers(target_spectrum, initial_thickness_nm, max_iter=10):
    """
    以 Beam Search 預測結果作為初始點，
    使用 L-BFGS-B 在連續厚度空間進行局部最佳化。

    Thickness Bounds:
        10.0 ~ 300.0 nm

    Output:
        optimized_thickness_nm : shape = (10,)
        optimized_spectrum     : shape = (601,)
        optimized_spectrum_mae : float
    """

    initial_x = initial_thickness_nm.astype(np.float64)
    bounds = [(10.0, 300.0)] * 10

    def loss_function(thickness_nm):
        simulated_spectrum = calculate_transmittance_spectrum(thickness_nm)
        return float(np.mean((target_spectrum - simulated_spectrum) ** 2))

    result = minimize(
        loss_function,
        x0=initial_x,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "eps": 1e-2,
            "maxiter": max_iter,
            "ftol": 1e-12,
            "gtol": 1e-7,
        },
    )

    optimized_thickness_nm = result.x.astype(np.float32)
    optimized_spectrum = calculate_transmittance_spectrum(optimized_thickness_nm)

    optimized_spectrum_mae = float(
        np.mean(np.abs(target_spectrum - optimized_spectrum))
    )

    return optimized_thickness_nm, optimized_spectrum, optimized_spectrum_mae

In [ ]:
# ==============================================================================
# Block 5: Single Evaluation + Batch Testing
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt


MODEL_PATH = "thin_film_transformer_final.pth"
BEAM_WIDTH = 24
LBFGSB_MAX_ITER = 10
SUCCESS_THRESHOLD = 0.03

model, model_config = load_trained_model(MODEL_PATH)


def evaluate_single_10_layer(target_thickness_nm, target_spectrum, plot=True):
    """
    對單一 10 層結構執行：

    Beam Search
        ↓
    TMM Candidate Selection
        ↓
    L-BFGS-B Continuous Refinement

    主要評估指標為 Transmittance Spectrum MAE。
    Thickness MAE 僅作為參考。
    """

    beam_thickness_nm, beam_spectrum, beam_mae = run_beam_search_10_layers(
        model,
        target_spectrum,
        beam_width=BEAM_WIDTH,
    )

    optimized_thickness_nm, optimized_spectrum, optimized_mae = run_lbfgsb_10_layers(
        target_spectrum,
        beam_thickness_nm,
        max_iter=LBFGSB_MAX_ITER,
    )

    beam_thickness_mae = float(
        np.mean(np.abs(beam_thickness_nm - target_thickness_nm))
    )

    optimized_thickness_mae = float(
        np.mean(np.abs(optimized_thickness_nm - target_thickness_nm))
    )

    if plot:

        print("=" * 76)
        print(
            f"{'Layer':<8}"
            f"{'Ground Truth':<20}"
            f"{'Beam Search':<20}"
            f"{'L-BFGS-B':<20}"
        )
        print("=" * 76)

        for i in range(10):
            print(
                f"{i + 1:<8}"
                f"{target_thickness_nm[i]:<20.2f}"
                f"{beam_thickness_nm[i]:<20.2f}"
                f"{optimized_thickness_nm[i]:<20.2f}"
            )

        print("=" * 76)

        print(f"Beam Search Spectrum MAE : {beam_mae:.6f}")
        print(f"L-BFGS-B Spectrum MAE    : {optimized_mae:.6f}")
        print(f"Beam Search Thickness MAE: {beam_thickness_mae:.2f} nm")
        print(f"L-BFGS-B Thickness MAE   : {optimized_thickness_mae:.2f} nm")

        plt.figure(figsize=(10, 4.8))

        plt.plot(
            WL_FULL,
            target_spectrum,
            label="Target",
            linewidth=2.2,
        )

        plt.plot(
            WL_FULL,
            beam_spectrum,
            label=f"Beam Search (MAE = {beam_mae:.4f})",
            linestyle="--",
        )

        plt.plot(
            WL_FULL,
            optimized_spectrum,
            label=f"L-BFGS-B (MAE = {optimized_mae:.4f})",
            linestyle=":",
        )

        plt.title("10-Layer Thin-Film Inverse Design")
        plt.xlabel("Wavelength (nm)")
        plt.ylabel("Transmittance")
        plt.ylim(-0.05, 1.05)
        plt.grid(True, linestyle=":")
        plt.legend()
        plt.tight_layout()
        plt.show()

    return {
        "beam_spectrum_mae": beam_mae,
        "optimized_spectrum_mae": optimized_mae,
        "beam_thickness_mae": beam_thickness_mae,
        "optimized_thickness_mae": optimized_thickness_mae,
    }


def run_batch_test(num_runs=1000, plot_boxplot=True):

    beam_maes = []
    optimized_maes = []

    beam_thickness_maes = []
    optimized_thickness_maes = []

    print(f"Running {num_runs} independent test samples...")

    for i in range(num_runs):

        target_thickness_nm, target_spectrum = generate_random_test_sample()

        result = evaluate_single_10_layer(
            target_thickness_nm,
            target_spectrum,
            plot=False,
        )

        beam_maes.append(result["beam_spectrum_mae"])
        optimized_maes.append(result["optimized_spectrum_mae"])

        beam_thickness_maes.append(result["beam_thickness_mae"])
        optimized_thickness_maes.append(result["optimized_thickness_mae"])

        if (i + 1) % max(1, num_runs // 10) == 0:
            print(f"Progress: {i + 1}/{num_runs}")


    beam_maes = np.asarray(beam_maes)
    optimized_maes = np.asarray(optimized_maes)

    beam_thickness_maes = np.asarray(beam_thickness_maes)
    optimized_thickness_maes = np.asarray(optimized_thickness_maes)


    failure_mask = optimized_maes > SUCCESS_THRESHOLD

    failure_rate = np.mean(failure_mask) * 100
    success_rate = 100 - failure_rate


    print("\n" + "=" * 68)
    print(f"{'Metric':<30}{'Beam Search':<18}{'L-BFGS-B':<18}")
    print("=" * 68)

    print(
        f"{'Spectrum MAE':<30}"
        f"{np.mean(beam_maes):.4f} ± {np.std(beam_maes):.4f}   "
        f"{np.mean(optimized_maes):.4f} ± {np.std(optimized_maes):.4f}"
    )

    print(
        f"{'Thickness MAE (nm)':<30}"
        f"{np.mean(beam_thickness_maes):.2f} ± {np.std(beam_thickness_maes):.2f}   "
        f"{np.mean(optimized_thickness_maes):.2f} ± {np.std(optimized_thickness_maes):.2f}"
    )

    print("=" * 68)

    print(f"\nSuccess threshold : Spectrum MAE ≤ {SUCCESS_THRESHOLD:.3f}")
    print(f"Success rate      : {success_rate:.2f}%")
    print(f"Failure rate      : {failure_rate:.2f}%")
    print(f"Failures          : {np.sum(failure_mask)}/{num_runs}")


    if plot_boxplot:

        plt.figure(figsize=(7, 4.5))

        plt.boxplot(
            [beam_maes, optimized_maes],
            tick_labels=["Beam Search", "L-BFGS-B"],
        )

        plt.axhline(
            SUCCESS_THRESHOLD,
            linestyle="--",
            label=f"Failure Threshold = {SUCCESS_THRESHOLD:.2f}",
        )

        plt.title("10-Layer Spectrum MAE")
        plt.ylabel("Transmittance MAE")
        plt.grid(True, linestyle=":")
        plt.legend()
        plt.tight_layout()
        plt.show()


    return {
        "beam_maes": beam_maes,
        "optimized_maes": optimized_maes,
        "beam_thickness_maes": beam_thickness_maes,
        "optimized_thickness_maes": optimized_thickness_maes,
        "success_rate": success_rate,
        "failure_rate": failure_rate,
    }

In [ ]:
# ==============================================================================
# Block 6: Single Test Run
# ==============================================================================

# 隨機生成一組全新的 10 層連續厚度結構
target_thickness_nm, target_spectrum = generate_random_test_sample()

# 執行 Beam Search + L-BFGS-B
result = evaluate_single_10_layer(
    target_thickness_nm,
    target_spectrum,
    plot=True,
)

beam_mae = result["beam_spectrum_mae"]
optimized_mae = result["optimized_spectrum_mae"]

beam_thickness_mae = result["beam_thickness_mae"]
optimized_thickness_mae = result["optimized_thickness_mae"]

# 計算改善幅度
spectrum_improvement = (
    (beam_mae - optimized_mae) / beam_mae * 100
    if beam_mae > 0 else 0.0
)

thickness_improvement = (
    (beam_thickness_mae - optimized_thickness_mae) / beam_thickness_mae * 100
    if beam_thickness_mae > 0 else 0.0
)

print("\n" + "=" * 68)
print(
    f"Beam Search | "
    f"Spectrum MAE: {beam_mae:.6f} | "
    f"Thickness MAE: {beam_thickness_mae:.2f} nm"
)

print(
    f"L-BFGS-B    | "
    f"Spectrum MAE: {optimized_mae:.6f} | "
    f"Thickness MAE: {optimized_thickness_mae:.2f} nm"
)

print(
    f"Spectrum Improvement : {spectrum_improvement:.2f}%\n"
    f"Thickness Improvement: {thickness_improvement:.2f}%"
)
print("=" * 68)

In [ ]:
# ==============================================================================
# Block 7: Batch Evaluation
# ==============================================================================

# 批量生成 1000 組全新測試結構，統計 Spectrum MAE 與 Thickness MAE
run_batch_test(num_runs=1000, plot_boxplot=True)